# Gaze-driven VLM input on SmolVLM2-2.2B: Full gate (when) + foveated tokens (where)

The same pipeline as the other VLM notebooks, on **SmolVLM2-2.2B-Instruct** (SigLIP + SmolLM2-1.7B),
the smallest model: one 384 px image per frame (as SmolVLM2 does for video frames), 14 px patches
on a 27 x 27 grid, a 3 x 3 pixel shuffle: **81 tokens per frame**, one token = one 42 px cell.

```
gaze + IMU -> Full gate (eye + head model) -> send policy (POLICY, default D+P): which frames are sent
sent frame (SmolVLM's own 384 px input) + gaze -> foveated cells BEFORE SigLIP: 42 px cells near
     the gaze, 126 px cells shrunk to 42 px elsewhere (SmolVLM's 3x steps); each cell = 3 x 3
     patches of 14 px with SmolVLM's own position ids
  -> SigLIP (full attention, its own weights) -> SmolVLM's pixel shuffle + projection
  -> one LLM token per cell -> SmolLM2                                      (no training)
```

SmolVLM's token grid is coarse (9 x 9, 3 x 3 cells of 126 px), so the FOVI rings of the other
models would split either one coarse cell or all nine. Instead **the k coarse cells nearest the gaze
stay at full resolution**: 9 + 8 k tokens, k = 1 / 3 / 5 / 7, about the same shares of the stock
input as the other models' four budgets.

| condition (all on the gate's frames) | tokens per frame |
|---|---|
| SmolVLM stock (384 px) | 81 |
| **fovea at gaze**, k = 1 / 3 / 5 / 7 | 17 / 33 / 49 / 65 |
| uniform: SmolVLM's own input downscaled to the closest token grid | 16 / 36 / 49 / 64 |
| fovea at image centre, k = 5 | 49 |

SmolVLM2's LLM context is 8k tokens and it was trained on at most 64 video frames, so questions with
more sent frames get them **evenly subsampled to 64** (the same frames in every condition). The
model (`HuggingFaceTB/SmolVLM2-2.2B-Instruct`, transformers-native) is not gated.

## Run order

1. Clone, mount, install, model check  2. AEA links  3. Questions  4. Full gate + send policy
5. Frames and gaze  6. SmolVLM: unit tests, parity with SmolVLM's own pipeline  7. Score  8. Results

## 1. Clone, mount, install, model check

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/smolvlm_fovea.py'), f'vlmeval/smolvlm_fovea.py is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest'], check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video and size, reused later
LOCAL_HI = '/content/frames_1344'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the Full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('Full model checkpoints: 5 folds found in', CKPT_FULL)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# SmolVLM2-2.2B-Instruct (transformers-native); a Hugging Face token is optional (rate limits)
from huggingface_hub import login, model_info
from vlmeval import smolvlm_fovea as sm
try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        login(token=_tok)
except Exception:
    pass
MODEL_ID = None
for cand in sm.MODEL_CANDIDATES:
    try:
        model_info(cand)
        MODEL_ID = cand
        break
    except Exception as ex:
        print(f'   {cand}: not available ({type(ex).__name__})')
assert MODEL_ID, f'none of {sm.MODEL_CANDIDATES} is reachable'
import transformers
print('model:', MODEL_ID, '| transformers', transformers.__version__)
assert hasattr(transformers, 'SmolVLMForConditionalGeneration'), \
    'this transformers has no SmolVLM (needs >= 4.50): pip install -U transformers, then restart the runtime'

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

# DEMO = True: only the 4 locked TEST videos (204 questions), to check that everything works.
# DEMO = False: all 12 gate videos with EgoEverything questions (834 questions).
DEMO = True
qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
if DEMO:
    qs = qs[qs.video.isin(test_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Full gate + policy D: which frames are sent

Each training video is scored by the fold model that held it out; test videos by the 5-fold
ensemble (cached on Drive). The held-out 4-class accuracy should be close to the
cross-validation numbers (~0.6), not higher, which would mean a video leaked into its own
training. Policy D then sends a frame when the view settles after motion, and on each
REFIXATION.

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"held-out 4-class accuracy on these {len(videos)} videos: {(o12.pred == o12.quad).mean():.3f}")

labels = {'full': egoev.video_labels(clips, oof)}
# Temporal policy: 'D' = send when the view settles + on each REFIXATION;
# 'D+P' = D + the first frame of every PURSUIT run; 'D+P all' = D + every PURSUIT frame.
POLICY = 'D+P'
POLICY_TAG = {'D': 'D', 'D+P': 'DP', 'D+P all': 'DPall'}[POLICY]   # separate result files per policy
sel = {'D: full': egoev.select_conditions(clips, labels, main='full')[f'{POLICY}: full']}
D = sel['D: full']
per = pd.DataFrame({'video_s': n_frames, 'frames sent': {v: len(D[v]) for v in videos}})
per['one frame every (s)'] = (per.video_s / per['frames sent']).round(1)
display(per)
print(f"policy {POLICY} sends {60 * per['frames sent'].sum() / per.video_s.sum():.1f} frames/min "
      f"({per['frames sent'].mean():.1f} per video)")

## 5. Frames and gaze

The 1344 px frames cached on Drive by the Qwen notebook are reused; SmolVLM's own processor turns
each into its 384 px input (no tiling), and every condition starts from that image. Videos not
cached yet are downloaded once. One gaze point per sent frame: the densest fixation cluster in the
second before it (`vlmeval.spatial.gaze_fixation`).

In [ ]:
from vlmeval.frames import ensure_frames, frame_path
from vlmeval import qwen_fovea as qf, spatial as sp

metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_HI, FRAMES_TAR, size=qf.IMG)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  {flag}")

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

tracks, n_drop = sp.gaze_tracks(FOLDS, videos, frame_time)
frames_of = {}
for v in videos:
    m = metas[v]
    frames_of[v] = []
    for f in D[v]:
        if f >= m['n']:
            continue
        t = float(m['ts'][f])
        gz_ = sp.gaze_fixation(tracks[v], t) if v in tracks else None
        frames_of[v].append(dict(key=(v, int(f)), path=frame_path(m, f), t=t,
                                 gaze=tuple(map(float, gz_)) if gz_ is not None else None))
items = [dict(qid=int(q.qid), video=q.video, question=q.question, options=list(q.options),
              answer=q.answer, frames=frames_of[q.video])
         for q in qs.sort_values(['video', 'qid']).itertuples()]
nf = np.array([len(it['frames']) for it in items])
print(f"{len(items)} questions on {len(videos)} videos; frames per question: mean {nf.mean():.1f}, max {nf.max()}; "
      f"gaze missing on {sum(fr['gaze'] is None for v in videos for fr in frames_of[v])} of "
      f"{sum(len(frames_of[v]) for v in videos)} sent frames")

## 6. SmolVLM2-2.2B: unit tests and parity with SmolVLM's own pipeline

| check | pass |
|---|---|
| unit tests (CPU, a tiny random SmolVLM from transformers' own classes on the real 27 x 27 grid): layouts (9 + 8 k tokens, gaze cell at full resolution), patch positions, patches = the image's 14 px patches, **all-42 px layout through our pre-encoder path = SmolVLM's own encoder + connector**, tokens follow their cells, uniform token counts, **our LLM inputs = the stock forward's logits**, prompt expansion, conditions; plus the Qwen2.5-VL tests (shared helpers) | all pass |
| real model, one frame: **vision in fp32** (all-42 px layout through our path vs SmolVLM's own encoder + connector on the processor's pixels) and **LLM** (our prompt carrying SmolVLM's own image tokens vs the processor's one-call API + model) | fp32 cosine > 0.9999; identical token ids and probabilities |
| our full path in bf16 (info) | deviation comparable to SmolVLM's own bf16-vs-fp32 deviation; same answer |
| feature drift: cosine of foveated tokens with SmolVLM's own tokens over the same area | 42 px cells close to 1 |

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_smolvlm_fovea.py', 'tests/test_qwen_fovea.py',
                    'tests/test_gaze_fixation.py', '-q', '-rs'], capture_output=True, text=True, cwd=REPO)
import transformers
print('transformers', transformers.__version__, '| torch', torch.__version__)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed: fix before using the GPU'

from PIL import Image
from vlmeval import qwen_mcq, fovea_registry as reg      # same prompt (frame times, question, options)
model, proc, letter_ids = sm.load_model(MODEL_ID)
print(MODEL_ID, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU | answer tokens',
      proc.tokenizer.convert_ids_to_tokens(letter_ids))
ip = proc.image_processor

q0 = qs.iloc[0]
m0 = metas[q0.video]
k0 = int(np.argmin(np.abs(np.array(m0['ts']) - q0.t0)))
pil0 = Image.open(frame_path(m0, k0)).convert('RGB')
rep = sm.parity_report(model, proc, letter_ids, pil0, q0.question, q0.options, qwen_mcq.build_messages)
print("parity, all-42 px layout through our path vs SmolVLM's processor + model:")
for k_, v_ in rep.items():
    print(f'   {k_:<38} {v_}')
# vision checked in fp32 (no bf16 rounding), LLM checked with SmolVLM's own image tokens
ok = (rep['images_in_stock'] == 1 and rep['pixel_max_abs_diff'] < 1e-3 and rep['own_path_vs_get_image_features'] < 1e-3
      and rep['vision_fp32_cos_min'] > 0.9999 and rep['tokens_ids_equal']
      and rep['llm_probs_max_abs_diff'] < 1e-3 and rep['llm_same_answer'])
assert ok, 'parity failed: do not trust the foveated results'
print('parity OK')

# SmolVLM2: <= 64 frames per question (8k context, trained on <= 64 video frames); the same frames in every condition
MAX_FRAMES = 64
def cap(frs):
    if len(frs) <= MAX_FRAMES:
        return frs
    return [frs[i] for i in np.unique(np.linspace(0, len(frs) - 1, MAX_FRAMES).round().astype(int))]
items_s = [dict(it, frames=cap(it['frames'])) for it in items]
n_cap = sum(len(a['frames']) < len(b['frames']) for a, b in zip(items_s, items))
print(f'\n{n_cap} of {len(items)} questions had more than {MAX_FRAMES} sent frames: evenly subsampled to {MAX_FRAMES}')

PICKS = reg.SMOL_K          # k of the 9 coarse cells at full resolution: 9 + 8 k tokens of 81
p_mid = PICKS[len(PICKS) // 2]
budget = pd.DataFrame({p['label']: {'k': p['k'], 'tokens per frame': 9 + 8 * p['k'],
                                    'uniform tokens': (sm.same_tokens_size(9 + 8 * p['k']) // sm.BASE) ** 2}
                       for p in PICKS}).T
budget['saving vs 81'] = (100 * (1 - budget['tokens per frame'] / 81)).round(1).astype(str) + '%'
print('token budgets (SmolVLM stock = 81 per 384 px frame):'); display(budget)

g0 = sp.gaze_fixation(tracks[q0.video], m0['ts'][k0]) or (0.5, 0.5)
rec0 = sm.stock_input(ip, pil0)
cells0 = sm.foveated_cells(*g0, p_mid['k'])
print(f"feature drift on one frame ({len(cells0)} tokens, k={p_mid['k']}):")
display(sm.fovea_feature_drift(model, rec0, cells0, ip.image_mean, ip.image_std).round(3))

COLS = {42: 'white', 126: 'deepskyblue'}
v_ = videos[0]
fr_ = frames_of[v_][len(frames_of[v_]) // 2]
im_ = sm.stock_input(ip, Image.open(fr_['path']).convert('RGB'))['x01'].clamp(0, 1).permute(1, 2, 0).numpy()
g_ = fr_['gaze'] or (0.5, 0.5)
fig, axs = plt.subplots(1, len(PICKS) + 1, figsize=(3.6 * (len(PICKS) + 1), 3.9))
axs[0].imshow(im_); axs[0].set_title('SmolVLM stock: 81 tokens', fontsize=8); axs[0].axis('off')
for a_, p in zip(axs[1:], PICKS):
    cells = sm.foveated_cells(*g_, p['k'])
    a_.imshow(im_)
    for y, x, s in cells:
        a_.add_patch(mpatches.Rectangle((x, y), s, s, fill=False, ec=COLS[s], lw=0.8))
    a_.plot(g_[0] * sm.IMG, g_[1] * sm.IMG, 'r+', ms=12, mew=2)
    a_.set_title(f"{len(cells)} tokens (k={p['k']})", fontsize=8); a_.axis('off')
plt.suptitle("one token per box: white 42 px (= SmolVLM's own token), blue 126 px; + = gaze", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Score: the gate's frames, SmolVLM stock vs foveated vs uniform downscaling

Every question gets the frames the send policy sent on its video (at most 64, see above). Work runs
video by video and, within a video, condition by condition: each frame's SmolVLM input is made once
per video and its tokens once per (video, condition). Answer = argmax of the raw next-token logits
over A-E, as SmolVLM writes it after `Assistant:` (" A"). Resumable.

In [ ]:
fov, UNI, STOCK = reg.smol_fovea, reg.smol_uniform, reg.SMOL_STOCK
CONDS = {STOCK: ('uniform', sm.IMG)}
for p in PICKS:
    CONDS[fov(p)] = ('fovea', dict(k=p['k'], gaze='gaze'))
    CONDS[UNI(p)] = ('uniform_match', fov(p))
CONDS[fov(p_mid, 'centre')] = ('fovea', dict(k=p_mid['k'], gaze='centre'))
print(f'{len(items_s)} questions x {len(CONDS)} conditions:', *CONDS, sep='\n   ')

OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_{POLICY_TAG}_fovea.jsonl")
sm.run_questions(model, proc, letter_ids, items_s, CONDS, OUT, qwen_mcq.build_messages)

## 8. Results

- accuracy per condition (95% interval from resampling videos), tokens per frame, saving vs 81
- paired tests at each budget: fovea vs uniform at the closest tokens, fovea vs stock
- **one pre-specified test across budgets**: per question, the mean over the 4 fovea budgets minus
  the mean over the 4 matched uniform budgets
- accuracy vs tokens plot, and **all VLMs side by side** (same frames; budget levels 1-4 = the four
  ring settings, or SmolVLM's four k; a model shows up once its notebook has run with the same POLICY)

In [ ]:
res = egoev.load_results(OUT)
res = res[res.video.isin(videos) & res.condition.isin(CONDS)]
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} rows not scored, left out:'); display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy()
res['correct'] = res['correct'].astype(int)

acc = egoev.accuracy_table(res)
acc['tokens / frame'] = res.groupby('condition').n_image_tokens.mean() / acc['frames_per_q']
acc['saving vs 81'] = 100 * (1 - acc['tokens / frame'] / 81)
acc = acc.loc[[c for c in CONDS if c in acc.index]]
print(f'SmolVLM2-2.2B, {res.qid.nunique()} questions on {res.video.nunique()} videos, policy {POLICY} frames '
      f'(<= {MAX_FRAMES} per question):')
display(acc[['n', 'accuracy', 'ci_low', 'ci_high', 'tokens / frame', 'saving vs 81']].round(1))

rows = []
for p in PICKS:
    for a_, b_ in ((fov(p), UNI(p)), (fov(p), STOCK)):
        if {a_, b_} <= set(res.condition):
            rows.append(dict(budget=p['label'], **egoev.paired_test(res, a_, b_)))
if {fov(p_mid), fov(p_mid, 'centre')} <= set(res.condition):
    rows.append(dict(budget=p_mid['label'], **egoev.paired_test(res, fov(p_mid), fov(p_mid, 'centre'))))
pt = pd.DataFrame(rows)
print('paired on the same questions (diff = a - b, accuracy points; CI from resampling videos):')
display(pt[['budget', 'a', 'b', 'diff', 'ci_low', 'ci_high', 'a_only_right', 'b_only_right', 'p_mcnemar']].round(3))

w = res.pivot_table(index=['qid', 'video'], columns='condition', values='correct')
f_cols, u_cols = [fov(p) for p in PICKS], [UNI(p) for p in PICKS]
if set(f_cols + u_cols) <= set(w.columns):
    d = pd.DataFrame({'video': w.index.get_level_values('video'),
                      'diff': 100 * (w[f_cols].mean(1) - w[u_cols].mean(1)).to_numpy()})
    lo, hi = egoev.video_bootstrap(d, lambda t: t['diff'].mean())
    pos, neg = int((d['diff'] > 0).sum()), int((d['diff'] < 0).sum())
    try:
        from scipy.stats import binomtest
        p_sign = binomtest(pos, pos + neg, 0.5).pvalue
    except ImportError:
        p_sign = float('nan')
    print(f"\nACROSS BUDGETS: fovea - uniform = {d['diff'].mean():+.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}] "
          f"(videos resampled); questions better / worse with fovea: {pos} / {neg}, sign test p = {p_sign:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.4))
def curve(names, style, label):
    d_ = acc.loc[[n for n in names if n in acc.index]].sort_values('tokens / frame')
    ax.errorbar(d_['tokens / frame'], d_.accuracy, yerr=[d_.accuracy - d_.ci_low, d_.ci_high - d_.accuracy],
                fmt=style, capsize=3, label=label)
curve(f_cols, 'o-', 'SmolVLM: fovea at gaze')
curve(u_cols, 's--', 'SmolVLM: uniform downscaled, closest tokens')
curve([STOCK], 'k*', 'SmolVLM stock (81)')
curve([fov(p_mid, 'centre')], 'x', 'SmolVLM: fovea at image centre')
ax.set(xlabel='visual tokens per frame', ylabel='accuracy (%)',
       title=f'SmolVLM2-2.2B, policy {POLICY} frames, {res.qid.nunique()} questions (95% CI over videos)')
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

acc_all, tok_all = reg.side_by_side(VLM_DIR, POLICY_TAG, videos)
print(f"\nall VLMs run so far, policy {POLICY} frames (accuracy %; budget levels 1-4 smallest first):")
display(acc_all.round(1))
print('visual tokens per frame:'); display(tok_all.round(0))

---

## Reading it

| look at | meaning |
|---|---|
| section 6 parity | must pass, or nothing after it is trustworthy |
| **across-budgets test** | the single main result for this model |
| fovea vs stock 81 | the token saving on the smallest model |
| fovea at gaze vs centre | is it the gaze, or the centre bias of head-mounted video? |
| all-VLM table | does the effect hold from 2B to 4B and across input designs? |

## Caveats

- Training-free: SigLIP never saw shrunk cells next to full-resolution ones; section 6's drift table
  shows how far the features move.
- The layout is coarser than on the other models (only 9 + 8 k token counts are possible, and k
  replaces the FOVI rings); the uniform baseline matches the token count to the closest square grid
  (16 / 36 / 49 / 64 vs 17 / 33 / 49 / 65).
- At most 64 frames per question here (the other models get every sent frame).
- EgoEverything barely needs resolution with 2-4B models (see the other notebooks).